# PySpark Comprehensive Tutorial

A hands-on guide from fundamentals to advanced PySpark operations.

## Table of Contents

1. [Setup & SparkSession](#1-setup--sparksession)
2. [Creating DataFrames](#2-creating-dataframes)
3. [Schema & Data Types](#3-schema--data-types)
4. [Basic DataFrame Operations](#4-basic-dataframe-operations)
5. [Column Operations & Expressions](#5-column-operations--expressions)
6. [Aggregations](#6-aggregations)
7. [Joins](#7-joins)
8. [Window Functions](#8-window-functions)
9. [User Defined Functions (UDFs)](#9-user-defined-functions-udfs)
10. [Spark SQL](#10-spark-sql)
11. [Reading & Writing Data](#11-reading--writing-data)
12. [Caching & Persistence](#12-caching--persistence)
13. [Partitioning & Performance](#13-partitioning--performance)
14. [Structured Streaming Introduction](#14-structured-streaming-introduction)
15. [Cleanup](#15-cleanup)

---
## 1. Setup & SparkSession

SparkSession is the unified entry point for all Spark functionality. It combines the older SparkContext, SQLContext, and HiveContext into a single object.

In [1]:
# Install PySpark if not already installed
# !pip install pyspark

import warnings

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

warnings.filterwarnings("ignore")

In [ ]:
# Create SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PySpark Tutorial")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Python Version: {spark.sparkContext.pythonVer}")
print("Spark UI: http://localhost:4040")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/05 18:55:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.2.0
Python Version: 3.12
Spark UI: http://localhost:4040


---
## 2. Creating DataFrames

We'll create a multi-table e-commerce dataset that will be used throughout the tutorial.

### Tables:
- **customers** — Customer information
- **products** — Product catalog
- **orders** — Order headers
- **order_items** — Line items for each order

In [ ]:
# --- CUSTOMERS TABLE ---
customers_data = [
    (1, "Alice Johnson", "alice@email.com", "New York", "NY", "2020-01-15"),
    (2, "Bob Smith", "bob@email.com", "Los Angeles", "CA", "2020-03-22"),
    (3, "Charlie Brown", "charlie@email.com", "Chicago", "IL", "2020-06-10"),
    (4, "Diana Ross", "diana@email.com", "Houston", "TX", "2021-01-05"),
    (5, "Eve Wilson", "eve@email.com", "Phoenix", "AZ", "2021-04-18"),
    (6, "Frank Miller", "frank@email.com", "New York", "NY", "2021-07-30"),
    (7, "Grace Lee", "grace@email.com", "San Francisco", "CA", "2021-09-12"),
    (8, "Henry Davis", "henry@email.com", "Chicago", "IL", "2022-02-28"),
    (9, "Ivy Chen", "ivy@email.com", "Seattle", "WA", "2022-05-14"),
    (10, "Jack Thompson", "jack@email.com", "Boston", "MA", "2022-08-01"),
]

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("email", StringType(), False),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("signup_date", StringType(), True),
])

customers = spark.createDataFrame(customers_data, schema=customers_schema)
customers = customers.withColumn("signup_date", F.to_date("signup_date"))

print("=== CUSTOMERS ===")
customers.show(truncate=False)

=== CUSTOMERS ===
+-----------+-------------+-----------------+-------------+-----+-----------+
|customer_id|name         |email            |city         |state|signup_date|
+-----------+-------------+-----------------+-------------+-----+-----------+
|1          |Alice Johnson|alice@email.com  |New York     |NY   |2020-01-15 |
|2          |Bob Smith    |bob@email.com    |Los Angeles  |CA   |2020-03-22 |
|3          |Charlie Brown|charlie@email.com|Chicago      |IL   |2020-06-10 |
|4          |Diana Ross   |diana@email.com  |Houston      |TX   |2021-01-05 |
|5          |Eve Wilson   |eve@email.com    |Phoenix      |AZ   |2021-04-18 |
|6          |Frank Miller |frank@email.com  |New York     |NY   |2021-07-30 |
|7          |Grace Lee    |grace@email.com  |San Francisco|CA   |2021-09-12 |
|8          |Henry Davis  |henry@email.com  |Chicago      |IL   |2022-02-28 |
|9          |Ivy Chen     |ivy@email.com    |Seattle      |WA   |2022-05-14 |
|10         |Jack Thompson|jack@email.com   |B

In [ ]:
# --- PRODUCTS TABLE ---
products_data = [
    (101, "Laptop Pro 15", "Electronics", 1299.99, 50),
    (102, "Wireless Mouse", "Electronics", 29.99, 200),
    (103, "USB-C Hub", "Electronics", 49.99, 150),
    (104, "Standing Desk", "Furniture", 599.99, 30),
    (105, "Ergonomic Chair", "Furniture", 449.99, 45),
    (106, 'Monitor 27"', "Electronics", 399.99, 75),
    (107, "Mechanical Keyboard", "Electronics", 89.99, 120),
    (108, "Desk Lamp", "Furniture", 39.99, 200),
    (109, "Webcam HD", "Electronics", 69.99, 100),
    (110, "Noise Cancelling Headphones", "Electronics", 249.99, 80),
    (111, "Bookshelf", "Furniture", 129.99, 60),
    (112, "Whiteboard", "Office Supplies", 79.99, 40),
    (113, "Notebook Set", "Office Supplies", 12.99, 500),
    (114, "Pen Set", "Office Supplies", 8.99, 800),
    (115, "Cable Management Kit", "Office Supplies", 19.99, 300),
]

products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("stock_quantity", IntegerType(), True),
])

products = spark.createDataFrame(products_data, schema=products_schema)

print("=== PRODUCTS ===")
products.show(truncate=False)

=== PRODUCTS ===
+----------+---------------------------+---------------+-------+--------------+
|product_id|product_name               |category       |price  |stock_quantity|
+----------+---------------------------+---------------+-------+--------------+
|101       |Laptop Pro 15              |Electronics    |1299.99|50            |
|102       |Wireless Mouse             |Electronics    |29.99  |200           |
|103       |USB-C Hub                  |Electronics    |49.99  |150           |
|104       |Standing Desk              |Furniture      |599.99 |30            |
|105       |Ergonomic Chair            |Furniture      |449.99 |45            |
|106       |Monitor 27"                |Electronics    |399.99 |75            |
|107       |Mechanical Keyboard        |Electronics    |89.99  |120           |
|108       |Desk Lamp                  |Furniture      |39.99  |200           |
|109       |Webcam HD                  |Electronics    |69.99  |100           |
|110       |Noise Cance

In [ ]:
# --- ORDERS TABLE ---
orders_data = [
    (1001, 1, "2023-01-10", "completed", 1349.98),
    (1002, 2, "2023-01-15", "completed", 599.99),
    (1003, 1, "2023-02-20", "completed", 539.97),
    (1004, 3, "2023-03-05", "completed", 1749.98),
    (1005, 4, "2023-03-18", "cancelled", 89.99),
    (1006, 5, "2023-04-02", "completed", 329.97),
    (1007, 2, "2023-04-22", "completed", 449.99),
    (1008, 6, "2023-05-10", "completed", 1299.99),
    (1009, 3, "2023-05-28", "returned", 249.99),
    (1010, 7, "2023-06-15", "completed", 869.97),
    (1011, 1, "2023-07-01", "completed", 399.99),
    (1012, 8, "2023-07-20", "completed", 159.98),
    (1013, 4, "2023-08-05", "completed", 1949.97),
    (1014, 9, "2023-08-22", "completed", 79.98),
    (1015, 5, "2023-09-10", "completed", 599.99),
    (1016, 10, "2023-09-28", "cancelled", 1299.99),
    (1017, 2, "2023-10-15", "completed", 269.98),
    (1018, 6, "2023-10-30", "completed", 489.98),
    (1019, 7, "2023-11-12", "completed", 149.98),
    (1020, 3, "2023-11-25", "completed", 2099.97),
    (1021, 1, "2023-12-05", "completed", 339.98),
    (1022, 4, "2023-12-18", "completed", 699.98),
    (1023, 9, "2023-12-28", "completed", 1349.98),
    (1024, 8, "2024-01-05", "completed", 449.99),
    (1025, 5, "2024-01-20", "pending", 89.99),
]

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("total_amount", DoubleType(), True),
])

orders = spark.createDataFrame(orders_data, schema=orders_schema)
orders = orders.withColumn("order_date", F.to_date("order_date"))

print("=== ORDERS ===")
orders.show(truncate=False)

=== ORDERS ===
+--------+-----------+----------+---------+------------+
|order_id|customer_id|order_date|status   |total_amount|
+--------+-----------+----------+---------+------------+
|1001    |1          |2023-01-10|completed|1349.98     |
|1002    |2          |2023-01-15|completed|599.99      |
|1003    |1          |2023-02-20|completed|539.97      |
|1004    |3          |2023-03-05|completed|1749.98     |
|1005    |4          |2023-03-18|cancelled|89.99       |
|1006    |5          |2023-04-02|completed|329.97      |
|1007    |2          |2023-04-22|completed|449.99      |
|1008    |6          |2023-05-10|completed|1299.99     |
|1009    |3          |2023-05-28|returned |249.99      |
|1010    |7          |2023-06-15|completed|869.97      |
|1011    |1          |2023-07-01|completed|399.99      |
|1012    |8          |2023-07-20|completed|159.98      |
|1013    |4          |2023-08-05|completed|1949.97     |
|1014    |9          |2023-08-22|completed|79.98       |
|1015    |5     

In [ ]:
# --- ORDER ITEMS TABLE ---
order_items_data = [
    (1, 1001, 101, 1, 1299.99),
    (2, 1001, 103, 1, 49.99),
    (3, 1002, 104, 1, 599.99),
    (4, 1003, 106, 1, 399.99),
    (5, 1003, 107, 1, 89.99),
    (6, 1003, 103, 1, 49.99),
    (7, 1004, 101, 1, 1299.99),
    (8, 1004, 105, 1, 449.99),
    (9, 1005, 107, 1, 89.99),
    (10, 1006, 110, 1, 249.99),
    (11, 1006, 102, 1, 29.99),
    (12, 1006, 103, 1, 49.99),
    (13, 1007, 105, 1, 449.99),
    (14, 1008, 101, 1, 1299.99),
    (15, 1009, 110, 1, 249.99),
    (16, 1010, 106, 1, 399.99),
    (17, 1010, 107, 1, 89.99),
    (18, 1010, 109, 1, 69.99),
    (19, 1010, 115, 2, 19.99),
    (20, 1011, 106, 1, 399.99),
    (21, 1012, 108, 2, 39.99),
    (22, 1012, 112, 1, 79.99),
    (23, 1013, 101, 1, 1299.99),
    (24, 1013, 104, 1, 599.99),
    (25, 1013, 103, 1, 49.99),
    (26, 1014, 113, 3, 12.99),
    (27, 1014, 114, 5, 8.99),
    (28, 1015, 104, 1, 599.99),
    (29, 1016, 101, 1, 1299.99),
    (30, 1017, 110, 1, 249.99),
    (31, 1017, 115, 1, 19.99),
    (32, 1018, 106, 1, 399.99),
    (33, 1018, 107, 1, 89.99),
    (34, 1019, 102, 2, 29.99),
    (35, 1019, 107, 1, 89.99),
    (36, 1020, 101, 1, 1299.99),
    (37, 1020, 106, 1, 399.99),
    (38, 1020, 106, 1, 399.99),
    (39, 1021, 110, 1, 249.99),
    (40, 1021, 107, 1, 89.99),
    (41, 1022, 104, 1, 599.99),
    (42, 1022, 108, 1, 39.99),
    (43, 1022, 115, 3, 19.99),
    (44, 1023, 101, 1, 1299.99),
    (45, 1023, 103, 1, 49.99),
    (46, 1024, 105, 1, 449.99),
    (47, 1025, 107, 1, 89.99),
]

order_items_schema = StructType([
    StructField("item_id", IntegerType(), False),
    StructField("order_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
])

order_items = spark.createDataFrame(order_items_data, schema=order_items_schema)

print("=== ORDER ITEMS (first 15 rows) ===")
order_items.show(15, truncate=False)
print(f"Total order items: {order_items.count()}")

=== ORDER ITEMS (first 15 rows) ===
+-------+--------+----------+--------+----------+
|item_id|order_id|product_id|quantity|unit_price|
+-------+--------+----------+--------+----------+
|1      |1001    |101       |1       |1299.99   |
|2      |1001    |103       |1       |49.99     |
|3      |1002    |104       |1       |599.99    |
|4      |1003    |106       |1       |399.99    |
|5      |1003    |107       |1       |89.99     |
|6      |1003    |103       |1       |49.99     |
|7      |1004    |101       |1       |1299.99   |
|8      |1004    |105       |1       |449.99    |
|9      |1005    |107       |1       |89.99     |
|10     |1006    |110       |1       |249.99    |
|11     |1006    |102       |1       |29.99     |
|12     |1006    |103       |1       |49.99     |
|13     |1007    |105       |1       |449.99    |
|14     |1008    |101       |1       |1299.99   |
|15     |1009    |110       |1       |249.99    |
+-------+--------+----------+--------+----------+
only showing t

---
## 3. Schema & Data Types

Understanding schemas is critical for working with Spark DataFrames.

In [7]:
# Print schema (tree format)
print("=== Customers Schema ===")
customers.printSchema()

print("\n=== Orders Schema ===")
orders.printSchema()

=== Customers Schema ===
root
 |-- customer_id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- email: string (nullable = false)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)


=== Orders Schema ===
root
 |-- order_id: integer (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)



In [8]:
# Programmatic schema access
print("Column names:", customers.columns)
print("\nData types:", customers.dtypes)
print("\nNumber of rows:", customers.count())
print("Number of columns:", len(customers.columns))

Column names: ['customer_id', 'name', 'email', 'city', 'state', 'signup_date']

Data types: [('customer_id', 'int'), ('name', 'string'), ('email', 'string'), ('city', 'string'), ('state', 'string'), ('signup_date', 'date')]

Number of rows: 10
Number of columns: 6


In [9]:
# Describe — summary statistics for numeric and string columns
products.describe().show()

26/08/05 18:55:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+----------------+--------------+---------------+------------------+------------------+
|summary|      product_id|  product_name|       category|             price|    stock_quantity|
+-------+----------------+--------------+---------------+------------------+------------------+
|  count|              15|            15|             15|                15|                15|
|   mean|           108.0|          NULL|           NULL|235.45666666666654|183.33333333333334|
| stddev|4.47213595499958|          NULL|           NULL| 346.9542717227541| 211.3533489094462|
|    min|             101|     Bookshelf|    Electronics|              8.99|                30|
|    max|             115|Wireless Mouse|Office Supplies|           1299.99|               800|
+-------+----------------+--------------+---------------+------------------+------------------+



In [10]:
# Detailed summary (includes percentiles, null counts)
products.summary().show()

+-------+----------------+--------------+---------------+------------------+------------------+
|summary|      product_id|  product_name|       category|             price|    stock_quantity|
+-------+----------------+--------------+---------------+------------------+------------------+
|  count|              15|            15|             15|                15|                15|
|   mean|           108.0|          NULL|           NULL|235.45666666666654|183.33333333333334|
| stddev|4.47213595499958|          NULL|           NULL| 346.9542717227541| 211.3533489094462|
|    min|             101|     Bookshelf|    Electronics|              8.99|                30|
|    25%|             104|          NULL|           NULL|             29.99|                50|
|    50%|             108|          NULL|           NULL|             79.99|               100|
|    75%|             112|          NULL|           NULL|            399.99|               200|
|    max|             115|Wireless Mouse

---
## 4. Basic DataFrame Operations

### Select, Filter, Sort, Distinct

In [11]:
# SELECT — choose specific columns
print("=== Select specific columns ===")
customers.select("name", "city", "state").show(5)

# Select with column expressions
print("\n=== Select with expressions ===")
products.select(
    F.col("product_name"),
    F.col("price"),
    (F.col("price") * 0.9).alias("discounted_price"),
    F.col("category"),
).show(5)

=== Select specific columns ===
+-------------+-----------+-----+
|         name|       city|state|
+-------------+-----------+-----+
|Alice Johnson|   New York|   NY|
|    Bob Smith|Los Angeles|   CA|
|Charlie Brown|    Chicago|   IL|
|   Diana Ross|    Houston|   TX|
|   Eve Wilson|    Phoenix|   AZ|
+-------------+-----------+-----+
only showing top 5 rows

=== Select with expressions ===
+---------------+-------+------------------+-----------+
|   product_name|  price|  discounted_price|   category|
+---------------+-------+------------------+-----------+
|  Laptop Pro 15|1299.99|          1169.991|Electronics|
| Wireless Mouse|  29.99|            26.991|Electronics|
|      USB-C Hub|  49.99|            44.991|Electronics|
|  Standing Desk| 599.99|           539.991|  Furniture|
|Ergonomic Chair| 449.99|404.99100000000004|  Furniture|
+---------------+-------+------------------+-----------+
only showing top 5 rows


In [12]:
# FILTER / WHERE — filter rows based on conditions
print("=== Electronics products over $100 ===")
products.filter((F.col("category") == "Electronics") & (F.col("price") > 100)).show()

# Alternative syntax using where() — same as filter()
print("=== Customers in California ===")
customers.where(F.col("state") == "CA").show()

=== Electronics products over $100 ===
+----------+--------------------+-----------+-------+--------------+
|product_id|        product_name|   category|  price|stock_quantity|
+----------+--------------------+-----------+-------+--------------+
|       101|       Laptop Pro 15|Electronics|1299.99|            50|
|       106|         Monitor 27"|Electronics| 399.99|            75|
|       110|Noise Cancelling ...|Electronics| 249.99|            80|
+----------+--------------------+-----------+-------+--------------+

=== Customers in California ===
+-----------+---------+---------------+-------------+-----+-----------+
|customer_id|     name|          email|         city|state|signup_date|
+-----------+---------+---------------+-------------+-----+-----------+
|          2|Bob Smith|  bob@email.com|  Los Angeles|   CA| 2020-03-22|
|          7|Grace Lee|grace@email.com|San Francisco|   CA| 2021-09-12|
+-----------+---------+---------------+-------------+-----+-----------+



In [13]:
# Multiple filter conditions
print("=== Completed orders over $500 ===")
orders.filter(
    (F.col("status") == "completed")
    & (F.col("total_amount") > 500)
    & (F.col("order_date") >= "2023-06-01")
).orderBy(F.col("total_amount").desc()).show()

# Using isin() for multiple values
print("=== Customers in NY or CA ===")
customers.filter(F.col("state").isin("NY", "CA")).show()

=== Completed orders over $500 ===
+--------+-----------+----------+---------+------------+
|order_id|customer_id|order_date|   status|total_amount|
+--------+-----------+----------+---------+------------+
|    1020|          3|2023-11-25|completed|     2099.97|
|    1013|          4|2023-08-05|completed|     1949.97|
|    1023|          9|2023-12-28|completed|     1349.98|
|    1010|          7|2023-06-15|completed|      869.97|
|    1022|          4|2023-12-18|completed|      699.98|
|    1015|          5|2023-09-10|completed|      599.99|
+--------+-----------+----------+---------+------------+

=== Customers in NY or CA ===
+-----------+-------------+---------------+-------------+-----+-----------+
|customer_id|         name|          email|         city|state|signup_date|
+-----------+-------------+---------------+-------------+-----+-----------+
|          1|Alice Johnson|alice@email.com|     New York|   NY| 2020-01-15|
|          2|    Bob Smith|  bob@email.com|  Los Angeles|   

In [14]:
# SORT / ORDER BY
print("=== Products sorted by price (descending) ===")
products.orderBy(F.col("price").desc()).show(5)

# Multiple sort columns
print("=== Products sorted by category (asc), then price (desc) ===")
products.orderBy(F.col("category").asc(), F.col("price").desc()).show()

=== Products sorted by price (descending) ===
+----------+--------------------+-----------+-------+--------------+
|product_id|        product_name|   category|  price|stock_quantity|
+----------+--------------------+-----------+-------+--------------+
|       101|       Laptop Pro 15|Electronics|1299.99|            50|
|       104|       Standing Desk|  Furniture| 599.99|            30|
|       105|     Ergonomic Chair|  Furniture| 449.99|            45|
|       106|         Monitor 27"|Electronics| 399.99|            75|
|       110|Noise Cancelling ...|Electronics| 249.99|            80|
+----------+--------------------+-----------+-------+--------------+
only showing top 5 rows
=== Products sorted by category (asc), then price (desc) ===
+----------+--------------------+---------------+-------+--------------+
|product_id|        product_name|       category|  price|stock_quantity|
+----------+--------------------+---------------+-------+--------------+
|       101|       Laptop Pro

In [15]:
# DISTINCT & DROP DUPLICATES
print("=== Distinct states ===")
customers.select("state").distinct().show()

print("=== Distinct categories ===")
products.select("category").distinct().show()

# Drop duplicates on specific columns
print("=== Unique order statuses ===")
orders.select("status").dropDuplicates().show()

=== Distinct states ===
+-----+
|state|
+-----+
|   NY|
|   CA|
|   IL|
|   TX|
|   AZ|
|   WA|
|   MA|
+-----+

=== Distinct categories ===
+---------------+
|       category|
+---------------+
|    Electronics|
|      Furniture|
|Office Supplies|
+---------------+

=== Unique order statuses ===
+---------+
|   status|
+---------+
|completed|
|cancelled|
| returned|
|  pending|
+---------+



---
## 5. Column Operations & Expressions

### withColumn, when/otherwise, String functions, Date functions

In [ ]:
# withColumn — add or modify columns
products_enhanced = (
    products
    .withColumn("price_with_tax", F.round(F.col("price") * 1.08, 2))
    .withColumn("is_expensive", F.col("price") > 200)
    .withColumn("stock_value", F.round(F.col("price") * F.col("stock_quantity"), 2))
)

products_enhanced.show(truncate=False)

+----------+---------------------------+---------------+-------+--------------+--------------+------------+-----------+
|product_id|product_name               |category       |price  |stock_quantity|price_with_tax|is_expensive|stock_value|
+----------+---------------------------+---------------+-------+--------------+--------------+------------+-----------+
|101       |Laptop Pro 15              |Electronics    |1299.99|50            |1403.99       |true        |64999.5    |
|102       |Wireless Mouse             |Electronics    |29.99  |200           |32.39         |false       |5998.0     |
|103       |USB-C Hub                  |Electronics    |49.99  |150           |53.99         |false       |7498.5     |
|104       |Standing Desk              |Furniture      |599.99 |30            |647.99        |true        |17999.7    |
|105       |Ergonomic Chair            |Furniture      |449.99 |45            |485.99        |true        |20249.55   |
|106       |Monitor 27"                |

In [ ]:
# WHEN / OTHERWISE — conditional expressions (like CASE WHEN in SQL)
products_tiered = products.withColumn(
    "price_tier",
    F
    .when(F.col("price") < 50, "Budget")
    .when(F.col("price") < 200, "Mid-Range")
    .when(F.col("price") < 500, "Premium")
    .otherwise("Luxury"),
)

products_tiered.select("product_name", "price", "price_tier").show(truncate=False)

+---------------------------+-------+----------+
|product_name               |price  |price_tier|
+---------------------------+-------+----------+
|Laptop Pro 15              |1299.99|Luxury    |
|Wireless Mouse             |29.99  |Budget    |
|USB-C Hub                  |49.99  |Budget    |
|Standing Desk              |599.99 |Luxury    |
|Ergonomic Chair            |449.99 |Premium   |
|Monitor 27"                |399.99 |Premium   |
|Mechanical Keyboard        |89.99  |Mid-Range |
|Desk Lamp                  |39.99  |Budget    |
|Webcam HD                  |69.99  |Mid-Range |
|Noise Cancelling Headphones|249.99 |Premium   |
|Bookshelf                  |129.99 |Mid-Range |
|Whiteboard                 |79.99  |Mid-Range |
|Notebook Set               |12.99  |Budget    |
|Pen Set                    |8.99   |Budget    |
|Cable Management Kit       |19.99  |Budget    |
+---------------------------+-------+----------+



In [18]:
# STRING FUNCTIONS
print("=== String Operations ===")
customers.select(
    F.col("name"),
    F.upper(F.col("name")).alias("upper_name"),
    F.lower(F.col("email")).alias("lower_email"),
    F.split(F.col("name"), " ")[0].alias("first_name"),
    F.split(F.col("name"), " ")[1].alias("last_name"),
    F.length(F.col("name")).alias("name_length"),
    F.substring(F.col("email"), 1, 5).alias("email_prefix"),
).show(truncate=False)

=== String Operations ===
+-------------+-------------+-----------------+----------+---------+-----------+------------+
|name         |upper_name   |lower_email      |first_name|last_name|name_length|email_prefix|
+-------------+-------------+-----------------+----------+---------+-----------+------------+
|Alice Johnson|ALICE JOHNSON|alice@email.com  |Alice     |Johnson  |13         |alice       |
|Bob Smith    |BOB SMITH    |bob@email.com    |Bob       |Smith    |9          |bob@e       |
|Charlie Brown|CHARLIE BROWN|charlie@email.com|Charlie   |Brown    |13         |charl       |
|Diana Ross   |DIANA ROSS   |diana@email.com  |Diana     |Ross     |10         |diana       |
|Eve Wilson   |EVE WILSON   |eve@email.com    |Eve       |Wilson   |10         |eve@e       |
|Frank Miller |FRANK MILLER |frank@email.com  |Frank     |Miller   |12         |frank       |
|Grace Lee    |GRACE LEE    |grace@email.com  |Grace     |Lee      |9          |grace       |
|Henry Davis  |HENRY DAVIS  |henry

In [19]:
# DATE FUNCTIONS
print("=== Date Operations ===")
orders.select(
    F.col("order_id"),
    F.col("order_date"),
    F.year("order_date").alias("year"),
    F.month("order_date").alias("month"),
    F.dayofweek("order_date").alias("day_of_week"),
    F.quarter("order_date").alias("quarter"),
    F.date_format("order_date", "MMMM yyyy").alias("formatted"),
    F.datediff(F.current_date(), F.col("order_date")).alias("days_ago"),
).show(10, truncate=False)

=== Date Operations ===
+--------+----------+----+-----+-----------+-------+-------------+--------+
|order_id|order_date|year|month|day_of_week|quarter|formatted    |days_ago|
+--------+----------+----+-----+-----------+-------+-------------+--------+
|1001    |2023-01-10|2023|1    |3          |1      |January 2023 |1303    |
|1002    |2023-01-15|2023|1    |1          |1      |January 2023 |1298    |
|1003    |2023-02-20|2023|2    |2          |1      |February 2023|1262    |
|1004    |2023-03-05|2023|3    |1          |1      |March 2023   |1249    |
|1005    |2023-03-18|2023|3    |7          |1      |March 2023   |1236    |
|1006    |2023-04-02|2023|4    |1          |2      |April 2023   |1221    |
|1007    |2023-04-22|2023|4    |7          |2      |April 2023   |1201    |
|1008    |2023-05-10|2023|5    |4          |2      |May 2023     |1183    |
|1009    |2023-05-28|2023|5    |1          |2      |May 2023     |1165    |
|1010    |2023-06-15|2023|6    |5          |2      |June 2023   

In [20]:
# NULL HANDLING
# Create sample data with nulls
data_with_nulls = [
    (1, "Alice", 100.0),
    (2, "Bob", None),
    (3, None, 200.0),
    (4, "Diana", None),
    (5, "Eve", 150.0),
]
df_nulls = spark.createDataFrame(data_with_nulls, ["id", "name", "amount"])

print("=== Original ===")
df_nulls.show()

print("=== Filter nulls ===")
df_nulls.filter(F.col("amount").isNotNull()).show()

print("=== Fill nulls ===")
df_nulls.fillna({"name": "Unknown", "amount": 0.0}).show()

print("=== Coalesce (first non-null) ===")
df_nulls.withColumn("amount_clean", F.coalesce(F.col("amount"), F.lit(0.0))).show()

=== Original ===
+---+-----+------+
| id| name|amount|
+---+-----+------+
|  1|Alice| 100.0|
|  2|  Bob|  NULL|
|  3| NULL| 200.0|
|  4|Diana|  NULL|
|  5|  Eve| 150.0|
+---+-----+------+

=== Filter nulls ===
+---+-----+------+
| id| name|amount|
+---+-----+------+
|  1|Alice| 100.0|
|  3| NULL| 200.0|
|  5|  Eve| 150.0|
+---+-----+------+

=== Fill nulls ===
+---+-------+------+
| id|   name|amount|
+---+-------+------+
|  1|  Alice| 100.0|
|  2|    Bob|   0.0|
|  3|Unknown| 200.0|
|  4|  Diana|   0.0|
|  5|    Eve| 150.0|
+---+-------+------+

=== Coalesce (first non-null) ===
+---+-----+------+------------+
| id| name|amount|amount_clean|
+---+-----+------+------------+
|  1|Alice| 100.0|       100.0|
|  2|  Bob|  NULL|         0.0|
|  3| NULL| 200.0|       200.0|
|  4|Diana|  NULL|         0.0|
|  5|  Eve| 150.0|       150.0|
+---+-----+------+------------+



---
## 6. Aggregations

### groupBy, agg, pivot

In [21]:
# Basic aggregations
print("=== Order Statistics by Status ===")
orders.groupBy("status").agg(
    F.count("*").alias("order_count"),
    F.round(F.sum("total_amount"), 2).alias("total_revenue"),
    F.round(F.avg("total_amount"), 2).alias("avg_order_value"),
    F.round(F.min("total_amount"), 2).alias("min_order"),
    F.round(F.max("total_amount"), 2).alias("max_order"),
).orderBy(F.col("order_count").desc()).show()

=== Order Statistics by Status ===
+---------+-----------+-------------+---------------+---------+---------+
|   status|order_count|total_revenue|avg_order_value|min_order|max_order|
+---------+-----------+-------------+---------------+---------+---------+
|completed|         21|     16229.59|         772.84|    79.98|  2099.97|
|cancelled|          2|      1389.98|         694.99|    89.99|  1299.99|
| returned|          1|       249.99|         249.99|   249.99|   249.99|
|  pending|          1|        89.99|          89.99|    89.99|    89.99|
+---------+-----------+-------------+---------------+---------+---------+



In [22]:
# Multi-level groupBy
print("=== Product Sales by Category ===")
order_items.join(products, "product_id").groupBy("category").agg(
    F.count("*").alias("items_sold"),
    F.sum(F.col("quantity") * F.col("unit_price")).alias("total_revenue"),
    F.countDistinct("product_id").alias("unique_products"),
    F.round(F.avg("unit_price"), 2).alias("avg_price"),
).orderBy(F.col("total_revenue").desc()).show()

=== Product Sales by Category ===
+---------------+----------+-------------+---------------+---------+
|       category|items_sold|total_revenue|unique_products|avg_price|
+---------------+----------+-------------+---------------+---------+
|    Electronics|        32|     13539.67|              7|   422.18|
|      Furniture|         9|       3869.9|              3|   425.55|
|Office Supplies|         6|       283.85|              4|    26.99|
+---------------+----------+-------------+---------------+---------+



In [23]:
# Monthly revenue trend
print("=== Monthly Revenue (2023) ===")
orders.filter(
    (F.col("status") == "completed") & (F.year("order_date") == 2023)
).withColumn("month", F.month("order_date")).groupBy("month").agg(
    F.count("*").alias("orders"),
    F.round(F.sum("total_amount"), 2).alias("revenue"),
).orderBy("month").show()

=== Monthly Revenue (2023) ===
+-----+------+-------+
|month|orders|revenue|
+-----+------+-------+
|    1|     2|1949.97|
|    2|     1| 539.97|
|    3|     1|1749.98|
|    4|     2| 779.96|
|    5|     1|1299.99|
|    6|     1| 869.97|
|    7|     2| 559.97|
|    8|     2|2029.95|
|    9|     1| 599.99|
|   10|     2| 759.96|
|   11|     2|2249.95|
|   12|     3|2389.94|
+-----+------+-------+



In [24]:
# PIVOT — transform rows into columns
print("=== Orders Pivot: Status by Quarter ===")
orders.withColumn("quarter", F.concat(F.lit("Q"), F.quarter("order_date"))).groupBy(
    "quarter"
).pivot("status").agg(F.count(F.lit(1))).fillna(0).orderBy("quarter").show()

=== Orders Pivot: Status by Quarter ===
+-------+---------+---------+-------+--------+
|quarter|cancelled|completed|pending|returned|
+-------+---------+---------+-------+--------+
|     Q1|        1|        5|      1|       0|
|     Q2|        0|        4|      0|       1|
|     Q3|        1|        5|      0|       0|
|     Q4|        0|        7|      0|       0|
+-------+---------+---------+-------+--------+



In [25]:
# Collect aggregation functions
print("=== Products per Category (as list) ===")
products.groupBy("category").agg(
    F.collect_list("product_name").alias("products"),
    F.collect_set("product_name").alias("unique_products"),
    F.count("*").alias("count"),
).show(truncate=False)

=== Products per Category (as list) ===
+---------------+--------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------+-----+
|category       |products                                                                                                            |unique_products                                                                                                     |count|
+---------------+--------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------+-----+
|Electronics    |[Laptop Pro 15, Wireless Mouse, USB-C Hub, Monitor 27", Mechanical Keyboard, Webcam HD, Noise Cancelling Headphones]|[Mechanical Keyboard, USB-C Hub, Noise Cancelling He

---
## 7. Joins

Spark supports: `inner`, `left` (left_outer), `right` (right_outer), `full` (full_outer), `cross`, `left_semi`, `left_anti`

In [26]:
# INNER JOIN — only matching rows from both sides
print("=== Inner Join: Orders with Customer Names ===")
orders.join(customers, "customer_id", "inner").select(
    "order_id", "name", "order_date", "total_amount", "status"
).orderBy("order_date").show(10)

=== Inner Join: Orders with Customer Names ===
+--------+-------------+----------+------------+---------+
|order_id|         name|order_date|total_amount|   status|
+--------+-------------+----------+------------+---------+
|    1001|Alice Johnson|2023-01-10|     1349.98|completed|
|    1002|    Bob Smith|2023-01-15|      599.99|completed|
|    1003|Alice Johnson|2023-02-20|      539.97|completed|
|    1004|Charlie Brown|2023-03-05|     1749.98|completed|
|    1005|   Diana Ross|2023-03-18|       89.99|cancelled|
|    1006|   Eve Wilson|2023-04-02|      329.97|completed|
|    1007|    Bob Smith|2023-04-22|      449.99|completed|
|    1008| Frank Miller|2023-05-10|     1299.99|completed|
|    1009|Charlie Brown|2023-05-28|      249.99| returned|
|    1010|    Grace Lee|2023-06-15|      869.97|completed|
+--------+-------------+----------+------------+---------+
only showing top 10 rows


In [27]:
# LEFT JOIN — all rows from left, matching from right
# Find customers who have never ordered
print("=== Left Join: All Customers with Order Count ===")
customers.join(
    orders.groupBy("customer_id").agg(F.count("*").alias("order_count")),
    "customer_id",
    "left",
).select("name", "city", "order_count").fillna(0, subset=["order_count"]).orderBy(
    F.col("order_count").desc()
).show()

=== Left Join: All Customers with Order Count ===
+-------------+-------------+-----------+
|         name|         city|order_count|
+-------------+-------------+-----------+
|Alice Johnson|     New York|          4|
|    Bob Smith|  Los Angeles|          3|
|Charlie Brown|      Chicago|          3|
|   Diana Ross|      Houston|          3|
|   Eve Wilson|      Phoenix|          3|
| Frank Miller|     New York|          2|
|    Grace Lee|San Francisco|          2|
|  Henry Davis|      Chicago|          2|
|     Ivy Chen|      Seattle|          2|
|Jack Thompson|       Boston|          1|
+-------------+-------------+-----------+



In [28]:
# LEFT ANTI JOIN — rows in left that have NO match in right
# Customers who have NEVER placed an order
print("=== Left Anti: Customers with NO orders ===")
customers.join(orders, "customer_id", "left_anti").select(
    "customer_id", "name", "city"
).show()

# LEFT SEMI JOIN — rows in left that DO have a match in right (like EXISTS)
print("=== Left Semi: Customers WITH orders ===")
customers.join(orders, "customer_id", "left_semi").select(
    "customer_id", "name", "city"
).show()

=== Left Anti: Customers with NO orders ===
+-----------+----+----+
|customer_id|name|city|
+-----------+----+----+
+-----------+----+----+

=== Left Semi: Customers WITH orders ===
+-----------+-------------+-------------+
|customer_id|         name|         city|
+-----------+-------------+-------------+
|          1|Alice Johnson|     New York|
|          2|    Bob Smith|  Los Angeles|
|          3|Charlie Brown|      Chicago|
|          4|   Diana Ross|      Houston|
|          5|   Eve Wilson|      Phoenix|
|          6| Frank Miller|     New York|
|          7|    Grace Lee|San Francisco|
|          8|  Henry Davis|      Chicago|
|          9|     Ivy Chen|      Seattle|
|         10|Jack Thompson|       Boston|
+-----------+-------------+-------------+



In [ ]:
# MULTI-TABLE JOIN — Full order details
print("=== Complete Order Details ===")
full_details = (
    order_items
    .join(orders, "order_id")
    .join(customers, "customer_id")
    .join(products, "product_id")
    .select(
        "order_id",
        "name",
        "order_date",
        "product_name",
        "category",
        "quantity",
        "unit_price",
        (F.col("quantity") * F.col("unit_price")).alias("line_total"),
    )
)

full_details.orderBy("order_id", "product_name").show(15, truncate=False)

=== Complete Order Details ===
+--------+-------------+----------+---------------------------+-----------+--------+----------+----------+
|order_id|name         |order_date|product_name               |category   |quantity|unit_price|line_total|
+--------+-------------+----------+---------------------------+-----------+--------+----------+----------+
|1001    |Alice Johnson|2023-01-10|Laptop Pro 15              |Electronics|1       |1299.99   |1299.99   |
|1001    |Alice Johnson|2023-01-10|USB-C Hub                  |Electronics|1       |49.99     |49.99     |
|1002    |Bob Smith    |2023-01-15|Standing Desk              |Furniture  |1       |599.99    |599.99    |
|1003    |Alice Johnson|2023-02-20|Mechanical Keyboard        |Electronics|1       |89.99     |89.99     |
|1003    |Alice Johnson|2023-02-20|Monitor 27"                |Electronics|1       |399.99    |399.99    |
|1003    |Alice Johnson|2023-02-20|USB-C Hub                  |Electronics|1       |49.99     |49.99     |
|1004 

In [30]:
# CROSS JOIN — cartesian product (use carefully!)
print("=== Cross Join: All Category-State Combinations ===")
categories = products.select("category").distinct()
states = customers.select("state").distinct()

categories.crossJoin(states).orderBy("category", "state").show(10)

=== Cross Join: All Category-State Combinations ===
+-----------+-----+
|   category|state|
+-----------+-----+
|Electronics|   AZ|
|Electronics|   CA|
|Electronics|   IL|
|Electronics|   MA|
|Electronics|   NY|
|Electronics|   TX|
|Electronics|   WA|
|  Furniture|   AZ|
|  Furniture|   CA|
|  Furniture|   IL|
+-----------+-----+
only showing top 10 rows


---
## 8. Window Functions

Window functions perform calculations across a set of rows that are related to the current row — without collapsing rows like `groupBy`.

### Types:
- **Ranking**: `row_number()`, `rank()`, `dense_rank()`, `percent_rank()`, `ntile()`
- **Analytic**: `lag()`, `lead()`, `first()`, `last()`
- **Aggregate**: `sum()`, `avg()`, `count()`, `min()`, `max()` over a window

In [31]:
# RANKING — rank products by price within each category
window_cat_price = Window.partitionBy("category").orderBy(F.col("price").desc())

print("=== Product Rankings by Category ===")
products.withColumn("row_num", F.row_number().over(window_cat_price)).withColumn(
    "rank", F.rank().over(window_cat_price)
).withColumn("dense_rank", F.dense_rank().over(window_cat_price)).select(
    "category", "product_name", "price", "row_num", "rank", "dense_rank"
).orderBy("category", "row_num").show(truncate=False)

=== Product Rankings by Category ===
+---------------+---------------------------+-------+-------+----+----------+
|category       |product_name               |price  |row_num|rank|dense_rank|
+---------------+---------------------------+-------+-------+----+----------+
|Electronics    |Laptop Pro 15              |1299.99|1      |1   |1         |
|Electronics    |Monitor 27"                |399.99 |2      |2   |2         |
|Electronics    |Noise Cancelling Headphones|249.99 |3      |3   |3         |
|Electronics    |Mechanical Keyboard        |89.99  |4      |4   |4         |
|Electronics    |Webcam HD                  |69.99  |5      |5   |5         |
|Electronics    |USB-C Hub                  |49.99  |6      |6   |6         |
|Electronics    |Wireless Mouse             |29.99  |7      |7   |7         |
|Furniture      |Standing Desk              |599.99 |1      |1   |1         |
|Furniture      |Ergonomic Chair            |449.99 |2      |2   |2         |
|Furniture      |Bookshelf 

In [32]:
# LAG / LEAD — access previous/next rows
# Track order-to-order changes for each customer
window_customer_date = Window.partitionBy("customer_id").orderBy("order_date")

print("=== Customer Order History with Lag/Lead ===")
orders.filter(F.col("status") == "completed").withColumn(
    "prev_order_amount", F.lag("total_amount", 1).over(window_customer_date)
).withColumn(
    "next_order_amount", F.lead("total_amount", 1).over(window_customer_date)
).withColumn(
    "prev_order_date", F.lag("order_date", 1).over(window_customer_date)
).withColumn(
    "days_between_orders", F.datediff(F.col("order_date"), F.col("prev_order_date"))
).join(customers.select("customer_id", "name"), "customer_id").select(
    "name", "order_date", "total_amount", "prev_order_amount", "days_between_orders"
).orderBy("name", "order_date").show(20, truncate=False)

=== Customer Order History with Lag/Lead ===
+-------------+----------+------------+-----------------+-------------------+
|name         |order_date|total_amount|prev_order_amount|days_between_orders|
+-------------+----------+------------+-----------------+-------------------+
|Alice Johnson|2023-01-10|1349.98     |NULL             |NULL               |
|Alice Johnson|2023-02-20|539.97      |1349.98          |41                 |
|Alice Johnson|2023-07-01|399.99      |539.97           |131                |
|Alice Johnson|2023-12-05|339.98      |399.99           |157                |
|Bob Smith    |2023-01-15|599.99      |NULL             |NULL               |
|Bob Smith    |2023-04-22|449.99      |599.99           |97                 |
|Bob Smith    |2023-10-15|269.98      |449.99           |176                |
|Charlie Brown|2023-03-05|1749.98     |NULL             |NULL               |
|Charlie Brown|2023-11-25|2099.97     |1749.98          |265                |
|Diana Ross   |2023

In [ ]:
# RUNNING AGGREGATES — cumulative sum, running average
window_running = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

print("=== Cumulative Spending per Customer ===")
orders.filter(F.col("status") == "completed").withColumn(
    "cumulative_spend", F.round(F.sum("total_amount").over(window_running), 2)
).withColumn(
    "running_avg", F.round(F.avg("total_amount").over(window_running), 2)
).withColumn(
    "order_num",
    F.row_number().over(Window.partitionBy("customer_id").orderBy("order_date")),
).join(customers.select("customer_id", "name"), "customer_id").select(
    "name", "order_num", "order_date", "total_amount", "cumulative_spend", "running_avg"
).filter(F.col("customer_id").isin(1, 2, 3)).orderBy("name", "order_date").show(
    truncate=False
)

=== Cumulative Spending per Customer ===
+-------------+---------+----------+------------+----------------+-----------+
|name         |order_num|order_date|total_amount|cumulative_spend|running_avg|
+-------------+---------+----------+------------+----------------+-----------+
|Alice Johnson|1        |2023-01-10|1349.98     |1349.98         |1349.98    |
|Alice Johnson|2        |2023-02-20|539.97      |1889.95         |944.98     |
|Alice Johnson|3        |2023-07-01|399.99      |2289.94         |763.31     |
|Alice Johnson|4        |2023-12-05|339.98      |2629.92         |657.48     |
|Bob Smith    |1        |2023-01-15|599.99      |599.99          |599.99     |
|Bob Smith    |2        |2023-04-22|449.99      |1049.98         |524.99     |
|Bob Smith    |3        |2023-10-15|269.98      |1319.96         |439.99     |
|Charlie Brown|1        |2023-03-05|1749.98     |1749.98         |1749.98    |
|Charlie Brown|2        |2023-11-25|2099.97     |3849.95         |1924.98    |
+----------

In [34]:
# NTILE — divide rows into N roughly equal groups
print("=== Products divided into Price Quartiles ===")
window_price = Window.orderBy(F.col("price").desc())

products.withColumn("quartile", F.ntile(4).over(window_price)).select(
    "product_name", "price", "quartile"
).orderBy("quartile", F.col("price").desc()).show(truncate=False)

=== Products divided into Price Quartiles ===
+---------------------------+-------+--------+
|product_name               |price  |quartile|
+---------------------------+-------+--------+
|Laptop Pro 15              |1299.99|1       |
|Standing Desk              |599.99 |1       |
|Ergonomic Chair            |449.99 |1       |
|Monitor 27"                |399.99 |1       |
|Noise Cancelling Headphones|249.99 |2       |
|Bookshelf                  |129.99 |2       |
|Mechanical Keyboard        |89.99  |2       |
|Whiteboard                 |79.99  |2       |
|Webcam HD                  |69.99  |3       |
|USB-C Hub                  |49.99  |3       |
|Desk Lamp                  |39.99  |3       |
|Wireless Mouse             |29.99  |3       |
|Cable Management Kit       |19.99  |4       |
|Notebook Set               |12.99  |4       |
|Pen Set                    |8.99   |4       |
+---------------------------+-------+--------+



26/08/05 18:55:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/05 18:55:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/05 18:55:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/05 18:55:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/05 18:55:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [35]:
# Percent of total within group
print("=== Product Price as % of Category Total ===")
window_cat = Window.partitionBy("category")

products.withColumn("category_total", F.sum("price").over(window_cat)).withColumn(
    "pct_of_category", F.round(F.col("price") / F.col("category_total") * 100, 1)
).select(
    "category", "product_name", "price", "category_total", "pct_of_category"
).orderBy("category", F.col("price").desc()).show(truncate=False)

=== Product Price as % of Category Total ===
+---------------+---------------------------+-------+------------------+---------------+
|category       |product_name               |price  |category_total    |pct_of_category|
+---------------+---------------------------+-------+------------------+---------------+
|Electronics    |Laptop Pro 15              |1299.99|2189.9300000000003|59.4           |
|Electronics    |Monitor 27"                |399.99 |2189.9300000000003|18.3           |
|Electronics    |Noise Cancelling Headphones|249.99 |2189.9300000000003|11.4           |
|Electronics    |Mechanical Keyboard        |89.99  |2189.9300000000003|4.1            |
|Electronics    |Webcam HD                  |69.99  |2189.9300000000003|3.2            |
|Electronics    |USB-C Hub                  |49.99  |2189.9300000000003|2.3            |
|Electronics    |Wireless Mouse             |29.99  |2189.9300000000003|1.4            |
|Furniture      |Standing Desk              |599.99 |1219.96     

---
## 9. User Defined Functions (UDFs)

When built-in functions aren't enough, you can define custom functions.

### UDF Types (ordered by performance):
1. **Built-in functions** (best) — use whenever possible
2. **Pandas UDFs** (good) — vectorized, uses Apache Arrow
3. **Python UDFs** (slowest) — row-by-row serialization

In [36]:
# PYTHON UDF — simple but slow (avoids Catalyst optimization)
from pyspark.sql.functions import udf


@udf(returnType=StringType())
def categorize_spending(amount):
    """Categorize order amount into spending tiers."""
    if amount is None:
        return "Unknown"
    elif amount < 100:
        return "Low"
    elif amount < 500:
        return "Medium"
    elif amount < 1000:
        return "High"
    else:
        return "Very High"


print("=== Orders with Spending Tier (Python UDF) ===")
orders.withColumn("spending_tier", categorize_spending(F.col("total_amount"))).select(
    "order_id", "total_amount", "spending_tier"
).show(10)

=== Orders with Spending Tier (Python UDF) ===
+--------+------------+-------------+
|order_id|total_amount|spending_tier|
+--------+------------+-------------+
|    1001|     1349.98|    Very High|
|    1002|      599.99|         High|
|    1003|      539.97|         High|
|    1004|     1749.98|    Very High|
|    1005|       89.99|          Low|
|    1006|      329.97|       Medium|
|    1007|      449.99|       Medium|
|    1008|     1299.99|    Very High|
|    1009|      249.99|       Medium|
|    1010|      869.97|         High|
+--------+------------+-------------+
only showing top 10 rows


In [37]:
# PANDAS UDF (Vectorized) — much faster for large datasets
import pandas as pd
from pyspark.sql.functions import pandas_udf


@pandas_udf(DoubleType())
def normalize_price(price_series: pd.Series) -> pd.Series:
    """Min-max normalize prices to 0-1 range."""
    min_val = price_series.min()
    max_val = price_series.max()
    if max_val == min_val:
        return pd.Series([0.5] * len(price_series))
    return (price_series - min_val) / (max_val - min_val)


print("=== Normalized Prices (Pandas UDF) ===")
products.withColumn(
    "normalized_price", F.round(normalize_price(F.col("price")), 4)
).select("product_name", "price", "normalized_price").orderBy(
    F.col("price").desc()
).show(truncate=False)

=== Normalized Prices (Pandas UDF) ===
+---------------------------+-------+----------------+
|product_name               |price  |normalized_price|
+---------------------------+-------+----------------+
|Laptop Pro 15              |1299.99|0.5             |
|Standing Desk              |599.99 |0.5             |
|Ergonomic Chair            |449.99 |0.5             |
|Monitor 27"                |399.99 |0.5             |
|Noise Cancelling Headphones|249.99 |0.5             |
|Bookshelf                  |129.99 |0.5             |
|Mechanical Keyboard        |89.99  |0.5             |
|Whiteboard                 |79.99  |0.5             |
|Webcam HD                  |69.99  |0.5             |
|USB-C Hub                  |49.99  |0.5             |
|Desk Lamp                  |39.99  |0.5             |
|Wireless Mouse             |29.99  |0.5             |
|Cable Management Kit       |19.99  |1.0             |
|Notebook Set               |12.99  |0.5             |
|Pen Set                  

In [ ]:
# PANDAS UDF with GroupBy (grouped map)
from pyspark.sql.functions import PandasUDFType, pandas_udf

# Define output schema
zscore_schema = StructType([
    StructField("product_id", IntegerType()),
    StructField("product_name", StringType()),
    StructField("category", StringType()),
    StructField("price", DoubleType()),
    StructField("price_zscore", DoubleType()),
])


@pandas_udf(zscore_schema, PandasUDFType.GROUPED_MAP)
def compute_zscore(pdf: pd.DataFrame) -> pd.DataFrame:
    """Compute z-score of price within each category."""
    mean = pdf["price"].mean()
    std = pdf["price"].std()
    if std == 0 or pd.isna(std):
        pdf["price_zscore"] = 0.0
    else:
        pdf["price_zscore"] = ((pdf["price"] - mean) / std).round(3)
    return pdf[["product_id", "product_name", "category", "price", "price_zscore"]]


print("=== Price Z-Scores by Category (Grouped Pandas UDF) ===")
products.groupBy("category").apply(compute_zscore).orderBy(
    "category", F.col("price").desc()
).show(truncate=False)

=== Price Z-Scores by Category (Grouped Pandas UDF) ===
+----------+---------------------------+---------------+-------+------------+
|product_id|product_name               |category       |price  |price_zscore|
+----------+---------------------------+---------------+-------+------------+
|101       |Laptop Pro 15              |Electronics    |1299.99|2.168       |
|106       |Monitor 27"                |Electronics    |399.99 |0.191       |
|110       |Noise Cancelling Headphones|Electronics    |249.99 |-0.138      |
|107       |Mechanical Keyboard        |Electronics    |89.99  |-0.489      |
|109       |Webcam HD                  |Electronics    |69.99  |-0.533      |
|103       |USB-C Hub                  |Electronics    |49.99  |-0.577      |
|102       |Wireless Mouse             |Electronics    |29.99  |-0.621      |
|104       |Standing Desk              |Furniture      |599.99 |1.118       |
|105       |Ergonomic Chair            |Furniture      |449.99 |0.549       |
|111    

---
## 10. Spark SQL

Spark SQL lets you query DataFrames using standard SQL syntax. Performance is identical to the DataFrame API — both use the Catalyst optimizer.

In [39]:
# Register DataFrames as temporary SQL views
customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")

print("Views registered successfully!")
spark.sql("SHOW TABLES").show()

Views registered successfully!
+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|         |  customers|       true|
|         |order_items|       true|
|         |     orders|       true|
|         |   products|       true|
+---------+-----------+-----------+



In [40]:
# Basic SELECT with filtering
print("=== SQL: High-value completed orders ===")
spark.sql("""
    SELECT 
        o.order_id,
        c.name AS customer_name,
        o.order_date,
        o.total_amount
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.status = 'completed' AND o.total_amount > 1000
    ORDER BY o.total_amount DESC
""").show()

=== SQL: High-value completed orders ===
+--------+-------------+----------+------------+
|order_id|customer_name|order_date|total_amount|
+--------+-------------+----------+------------+
|    1020|Charlie Brown|2023-11-25|     2099.97|
|    1013|   Diana Ross|2023-08-05|     1949.97|
|    1004|Charlie Brown|2023-03-05|     1749.98|
|    1001|Alice Johnson|2023-01-10|     1349.98|
|    1023|     Ivy Chen|2023-12-28|     1349.98|
|    1008| Frank Miller|2023-05-10|     1299.99|
+--------+-------------+----------+------------+



In [41]:
# Subqueries and CTEs
print("=== SQL: Top Customer per State (using CTE) ===")
spark.sql("""
    WITH customer_spending AS (
        SELECT 
            c.customer_id,
            c.name,
            c.state,
            COALESCE(SUM(o.total_amount), 0) AS total_spent,
            COUNT(o.order_id) AS order_count
        FROM customers c
        LEFT JOIN orders o ON c.customer_id = o.customer_id 
            AND o.status = 'completed'
        GROUP BY c.customer_id, c.name, c.state
    ),
    ranked AS (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY state ORDER BY total_spent DESC) AS rn
        FROM customer_spending
    )
    SELECT name, state, total_spent, order_count
    FROM ranked
    WHERE rn = 1
    ORDER BY total_spent DESC
""").show()

=== SQL: Top Customer per State (using CTE) ===
+-------------+-----+-----------+-----------+
|         name|state|total_spent|order_count|
+-------------+-----+-----------+-----------+
|Charlie Brown|   IL|    3849.95|          2|
|   Diana Ross|   TX|    2649.95|          2|
|Alice Johnson|   NY|    2629.92|          4|
|     Ivy Chen|   WA|    1429.96|          2|
|    Bob Smith|   CA|    1319.96|          3|
|   Eve Wilson|   AZ|     929.96|          2|
|Jack Thompson|   MA|        0.0|          0|
+-------------+-----+-----------+-----------+



In [42]:
# Window functions in SQL
print("=== SQL: Running Total and Moving Average ===")
spark.sql("""
    SELECT
        order_date,
        total_amount,
        SUM(total_amount) OVER (
            ORDER BY order_date 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_total,
        ROUND(AVG(total_amount) OVER (
            ORDER BY order_date 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS moving_avg_3
    FROM orders
    WHERE status = 'completed'
    ORDER BY order_date
""").show(15)

=== SQL: Running Total and Moving Average ===
+----------+------------+------------------+------------+
|order_date|total_amount|     running_total|moving_avg_3|
+----------+------------+------------------+------------+
|2023-01-10|     1349.98|           1349.98|     1349.98|
|2023-01-15|      599.99|           1949.97|      974.99|
|2023-02-20|      539.97|           2489.94|      829.98|
|2023-03-05|     1749.98|           4239.92|      963.31|
|2023-04-02|      329.97|           4569.89|      873.31|
|2023-04-22|      449.99|           5019.88|      843.31|
|2023-05-10|     1299.99|           6319.87|      693.32|
|2023-06-15|      869.97|           7189.84|      873.32|
|2023-07-01|      399.99|           7589.83|      856.65|
|2023-07-20|      159.98|7749.8099999999995|      476.65|
|2023-08-05|     1949.97| 9699.779999999999|      836.65|
|2023-08-22|       79.98| 9779.759999999998|      729.98|
|2023-09-10|      599.99|10379.749999999998|      876.65|
|2023-10-15|      269.98|1

26/08/05 18:55:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/05 18:55:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/05 18:55:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/05 18:55:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/05 18:55:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [43]:
# Complex analytical query
print("=== SQL: Product Performance Report ===")
spark.sql("""
    SELECT
        p.product_name,
        p.category,
        p.price,
        COUNT(oi.item_id) AS times_ordered,
        SUM(oi.quantity) AS total_units_sold,
        ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_revenue,
        ROUND(AVG(oi.quantity * oi.unit_price), 2) AS avg_revenue_per_order,
        RANK() OVER (PARTITION BY p.category ORDER BY SUM(oi.quantity * oi.unit_price) DESC) AS category_rank
    FROM products p
    LEFT JOIN order_items oi ON p.product_id = oi.product_id
    LEFT JOIN orders o ON oi.order_id = o.order_id AND o.status = 'completed'
    GROUP BY p.product_id, p.product_name, p.category, p.price
    ORDER BY total_revenue DESC
""").show(truncate=False)

=== SQL: Product Performance Report ===
+---------------------------+---------------+-------+-------------+----------------+-------------+---------------------+-------------+
|product_name               |category       |price  |times_ordered|total_units_sold|total_revenue|avg_revenue_per_order|category_rank|
+---------------------------+---------------+-------+-------------+----------------+-------------+---------------------+-------------+
|Laptop Pro 15              |Electronics    |1299.99|7            |7               |9099.93      |1299.99              |1            |
|Standing Desk              |Furniture      |599.99 |4            |4               |2399.96      |599.99               |1            |
|Monitor 27"                |Electronics    |399.99 |6            |6               |2399.94      |399.99               |2            |
|Ergonomic Chair            |Furniture      |449.99 |3            |3               |1349.97      |449.99               |2            |
|Noise Cancelli

In [44]:
# CASE WHEN in SQL
print("=== SQL: Customer Segmentation ===")
spark.sql("""
    SELECT
        c.name,
        c.state,
        COUNT(o.order_id) AS orders,
        ROUND(COALESCE(SUM(o.total_amount), 0), 2) AS total_spent,
        CASE
            WHEN COALESCE(SUM(o.total_amount), 0) >= 3000 THEN 'VIP'
            WHEN COALESCE(SUM(o.total_amount), 0) >= 1000 THEN 'Regular'
            WHEN COALESCE(SUM(o.total_amount), 0) > 0 THEN 'Occasional'
            ELSE 'Inactive'
        END AS segment
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id AND o.status = 'completed'
    GROUP BY c.customer_id, c.name, c.state
    ORDER BY total_spent DESC
""").show()

=== SQL: Customer Segmentation ===
+-------------+-----+------+-----------+----------+
|         name|state|orders|total_spent|   segment|
+-------------+-----+------+-----------+----------+
|Charlie Brown|   IL|     2|    3849.95|       VIP|
|   Diana Ross|   TX|     2|    2649.95|   Regular|
|Alice Johnson|   NY|     4|    2629.92|   Regular|
| Frank Miller|   NY|     2|    1789.97|   Regular|
|     Ivy Chen|   WA|     2|    1429.96|   Regular|
|    Bob Smith|   CA|     3|    1319.96|   Regular|
|    Grace Lee|   CA|     2|    1019.95|   Regular|
|   Eve Wilson|   AZ|     2|     929.96|Occasional|
|  Henry Davis|   IL|     2|     609.97|Occasional|
|Jack Thompson|   MA|     0|        0.0|  Inactive|
+-------------+-----+------+-----------+----------+



---
## 11. Reading & Writing Data

Spark supports many data formats. We'll demonstrate CSV, JSON, and Parquet.

In [45]:
import os
import tempfile

# Create a temp directory for our output files
output_dir = os.path.join(tempfile.gettempdir(), "pyspark_tutorial_output")
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

Output directory: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/pyspark_tutorial_output


In [46]:
# WRITE CSV
csv_path = os.path.join(output_dir, "products_csv")
products.write.mode("overwrite").csv(csv_path, header=True)
print(f"Written CSV to: {csv_path}")

# READ CSV
products_from_csv = spark.read.csv(csv_path, header=True, inferSchema=True)
print("\n=== Read back from CSV ===")
products_from_csv.show(5)

Written CSV to: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/pyspark_tutorial_output/products_csv

=== Read back from CSV ===
+----------+--------------------+---------------+------+--------------+
|product_id|        product_name|       category| price|stock_quantity|
+----------+--------------------+---------------+------+--------------+
|       114|             Pen Set|Office Supplies|  8.99|           800|
|       115|Cable Management Kit|Office Supplies| 19.99|           300|
|       110|Noise Cancelling ...|    Electronics|249.99|            80|
|       107| Mechanical Keyboard|    Electronics| 89.99|           120|
|       113|        Notebook Set|Office Supplies| 12.99|           500|
+----------+--------------------+---------------+------+--------------+
only showing top 5 rows


In [47]:
# WRITE JSON
json_path = os.path.join(output_dir, "orders_json")
orders.write.mode("overwrite").json(json_path)
print(f"Written JSON to: {json_path}")

# READ JSON
orders_from_json = spark.read.json(json_path)
print("\n=== Read back from JSON ===")
orders_from_json.show(5)

Written JSON to: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/pyspark_tutorial_output/orders_json

=== Read back from JSON ===
+-----------+----------+--------+---------+------------+
|customer_id|order_date|order_id|   status|total_amount|
+-----------+----------+--------+---------+------------+
|          5|2023-09-10|    1015|completed|      599.99|
|         10|2023-09-28|    1016|cancelled|     1299.99|
|          3|2023-11-25|    1020|completed|     2099.97|
|          1|2023-12-05|    1021|completed|      339.98|
|          4|2023-12-18|    1022|completed|      699.98|
+-----------+----------+--------+---------+------------+
only showing top 5 rows


In [48]:
# WRITE PARQUET (recommended format — columnar, compressed, schema-preserving)
parquet_path = os.path.join(output_dir, "full_data_parquet")

# Partitioned write — creates subdirectories by category
full_details.write.mode("overwrite").partitionBy("category").parquet(parquet_path)

print(f"Written partitioned Parquet to: {parquet_path}")
print("\nDirectory structure:")
for item in sorted(os.listdir(parquet_path)):
    print(f"  {item}")

Written partitioned Parquet to: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/pyspark_tutorial_output/full_data_parquet

Directory structure:
  ._SUCCESS.crc
  _SUCCESS
  category=Electronics
  category=Furniture
  category=Office Supplies


In [49]:
# READ PARQUET with partition pruning
print("=== Read Parquet (Electronics only — partition pruning) ===")
electronics = spark.read.parquet(parquet_path).filter(
    F.col("category") == "Electronics"
)

electronics.show(5, truncate=False)
print(f"\nRows in Electronics partition: {electronics.count()}")

=== Read Parquet (Electronics only — partition pruning) ===
+--------+-------------+----------+-------------------+--------+----------+----------+-----------+
|order_id|name         |order_date|product_name       |quantity|unit_price|line_total|category   |
+--------+-------------+----------+-------------------+--------+----------+----------+-----------+
|1001    |Alice Johnson|2023-01-10|USB-C Hub          |1       |49.99     |49.99     |Electronics|
|1001    |Alice Johnson|2023-01-10|Laptop Pro 15      |1       |1299.99   |1299.99   |Electronics|
|1003    |Alice Johnson|2023-02-20|USB-C Hub          |1       |49.99     |49.99     |Electronics|
|1003    |Alice Johnson|2023-02-20|Mechanical Keyboard|1       |89.99     |89.99     |Electronics|
|1003    |Alice Johnson|2023-02-20|Monitor 27"        |1       |399.99    |399.99    |Electronics|
+--------+-------------+----------+-------------------+--------+----------+----------+-----------+
only showing top 5 rows

Rows in Electronics part

In [ ]:
# READ with explicit schema (faster than inferSchema for CSV)
explicit_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("stock_quantity", IntegerType(), True),
])

products_explicit = spark.read.csv(csv_path, header=True, schema=explicit_schema)
print("=== Read with explicit schema ===")
products_explicit.printSchema()
products_explicit.show(3)

=== Read with explicit schema ===
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- stock_quantity: integer (nullable = true)

+----------+--------------------+---------------+------+--------------+
|product_id|        product_name|       category| price|stock_quantity|
+----------+--------------------+---------------+------+--------------+
|       114|             Pen Set|Office Supplies|  8.99|           800|
|       115|Cable Management Kit|Office Supplies| 19.99|           300|
|       110|Noise Cancelling ...|    Electronics|249.99|            80|
+----------+--------------------+---------------+------+--------------+
only showing top 3 rows


---
## 12. Caching & Persistence

Caching stores DataFrames in memory (or disk) to avoid recomputation. Critical for iterative algorithms and interactive analysis.

### Storage Levels:
| Level | Description |
|-------|-------------|
| MEMORY_ONLY | Store in JVM heap (fastest, may OOM) |
| MEMORY_AND_DISK | Spill to disk if memory full (default for `.cache()`) |
| DISK_ONLY | Store only on disk |
| MEMORY_ONLY_SER | Serialized in memory (less space, more CPU) |
| OFF_HEAP | Store in off-heap memory |

In [ ]:
import time

from pyspark import StorageLevel

# Create a "heavy" DataFrame (simulating expensive computation)
heavy_df = (
    order_items
    .join(orders, "order_id")
    .join(customers, "customer_id")
    .join(products, "product_id")
    .withColumn("line_total", F.col("quantity") * F.col("unit_price"))
    .withColumn("year", F.year("order_date"))
    .withColumn("month", F.month("order_date"))
)

# Without caching — two separate actions both trigger full recomputation
start = time.time()
count1 = heavy_df.count()
_ = heavy_df.filter(F.col("category") == "Electronics").count()
uncached_time = time.time() - start

# With caching
heavy_df.cache()  # Mark for caching (lazy — cached on first action)

start = time.time()
count2 = heavy_df.count()  # First action triggers cache
_ = heavy_df.filter(F.col("category") == "Electronics").count()  # Uses cache
cached_time = time.time() - start

print(f"Without cache: {uncached_time:.3f}s")
print(f"With cache:    {cached_time:.3f}s")
print("\nCache is most beneficial with larger datasets and repeated operations.")
print(f"\nStorage level: {heavy_df.storageLevel}")
print(f"Is cached: {heavy_df.is_cached}")

Without cache: 0.206s
With cache:    0.209s

Cache is most beneficial with larger datasets and repeated operations.

Storage level: Disk Memory Deserialized 1x Replicated
Is cached: True


In [52]:
# Unpersist when done
heavy_df.unpersist()
print(f"Is cached after unpersist: {heavy_df.is_cached}")

# Using specific storage level
heavy_df.persist(StorageLevel.MEMORY_AND_DISK)
heavy_df.count()  # Trigger caching
print(f"\nStorage level: {heavy_df.storageLevel}")

# Clean up
heavy_df.unpersist()
print("Cleaned up cache.")

Is cached after unpersist: False

Storage level: Disk Memory Serialized 1x Replicated
Cleaned up cache.


---
## 13. Partitioning & Performance

Proper partitioning is key to Spark performance. Too few partitions = underutilized cores. Too many = overhead from task scheduling.

In [53]:
# Check current partitions
print(f"Orders partitions: {orders.rdd.getNumPartitions()}")
print(f"Products partitions: {products.rdd.getNumPartitions()}")
print(f"Order items partitions: {order_items.rdd.getNumPartitions()}")

Orders partitions: 14
Products partitions: 14
Order items partitions: 14


In [54]:
# REPARTITION — full shuffle, creates specified number of partitions
# Use when increasing partitions or when you need even distribution
orders_repartitioned = orders.repartition(4)
print(f"After repartition(4): {orders_repartitioned.rdd.getNumPartitions()} partitions")

# Repartition by column (hash partitioning)
orders_by_customer = orders.repartition(4, "customer_id")
print(
    f"After repartition by customer_id: {orders_by_customer.rdd.getNumPartitions()} partitions"
)

# COALESCE — no shuffle, reduces partitions (merges existing)
orders_coalesced = orders_repartitioned.coalesce(2)
print(f"After coalesce(2): {orders_coalesced.rdd.getNumPartitions()} partitions")

After repartition(4): 4 partitions
After repartition by customer_id: 4 partitions
After coalesce(2): 2 partitions


In [55]:
# EXPLAIN — view the query execution plan
print("=== Simple Query Plan ===")
orders.filter(F.col("status") == "completed").groupBy("customer_id").agg(
    F.sum("total_amount").alias("total")
).explain(True)

=== Simple Query Plan ===
== Parsed Logical Plan ==
'Aggregate ['customer_id], ['customer_id, 'sum('total_amount) AS total#4727]
+- Filter (status#50 = completed)
   +- Project [order_id#47, customer_id#48, to_date(order_date#49, None, Some(Europe/London), true) AS order_date#52, status#50, total_amount#51]
      +- LogicalRDD [order_id#47, customer_id#48, order_date#49, status#50, total_amount#51], false

== Analyzed Logical Plan ==
customer_id: int, total: double
Aggregate [customer_id#48], [customer_id#48, sum(total_amount#51) AS total#4727]
+- Filter (status#50 = completed)
   +- Project [order_id#47, customer_id#48, to_date(order_date#49, None, Some(Europe/London), true) AS order_date#52, status#50, total_amount#51]
      +- LogicalRDD [order_id#47, customer_id#48, order_date#49, status#50, total_amount#51], false

== Optimized Logical Plan ==
Aggregate [customer_id#48], [customer_id#48, sum(total_amount#51) AS total#4727]
+- Project [customer_id#48, total_amount#51]
   +- Filter 

In [56]:
# BROADCAST JOIN — avoid shuffle for small tables
print("=== Broadcast Join Plan ===")
# Without broadcast
print("--- Without broadcast ---")
orders.join(customers, "customer_id").explain()

print("\n--- With broadcast ---")
orders.join(F.broadcast(customers), "customer_id").explain()

=== Broadcast Join Plan ===
--- Without broadcast ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#48, order_id#47, order_date#52, status#50, total_amount#51, name#1, email#2, city#3, state#4, signup_date#6]
   +- SortMergeJoin [customer_id#48], [customer_id#0], Inner
      :- Sort [customer_id#48 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(customer_id#48, 8), ENSURE_REQUIREMENTS, [plan_id=7314]
      :     +- Project [order_id#47, customer_id#48, cast(order_date#49 as date) AS order_date#52, status#50, total_amount#51]
      :        +- Scan ExistingRDD[order_id#47,customer_id#48,order_date#49,status#50,total_amount#51]
      +- Sort [customer_id#0 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(customer_id#0, 8), ENSURE_REQUIREMENTS, [plan_id=7315]
            +- Project [customer_id#0, name#1, email#2, city#3, state#4, cast(signup_date#5 as date) AS signup_date#6]
               +- Scan ExistingRDD[customer_id#

In [ ]:
# Performance tip: Predicate pushdown and column pruning
print("=== Optimized Query: Spark pushes filters down ===")

# This query benefits from predicate pushdown when reading from Parquet
optimized = (
    spark.read
    .parquet(parquet_path)
    .filter(F.col("category") == "Electronics")
    .select("order_id", "name", "product_name", "line_total")
)

optimized.explain(True)
print("\nResult:")
optimized.show(5)

=== Optimized Query: Spark pushes filters down ===
== Parsed Logical Plan ==
'Project ['order_id, 'name, 'product_name, 'line_total]
+- Filter (category#4743 = Electronics)
   +- Relation [order_id#4736,name#4737,order_date#4738,product_name#4739,quantity#4740,unit_price#4741,line_total#4742,category#4743] parquet

== Analyzed Logical Plan ==
order_id: int, name: string, product_name: string, line_total: double
Project [order_id#4736, name#4737, product_name#4739, line_total#4742]
+- Filter (category#4743 = Electronics)
   +- Relation [order_id#4736,name#4737,order_date#4738,product_name#4739,quantity#4740,unit_price#4741,line_total#4742,category#4743] parquet

== Optimized Logical Plan ==
Project [order_id#4736, name#4737, product_name#4739, line_total#4742]
+- Filter (isnotnull(category#4743) AND (category#4743 = Electronics))
   +- Relation [order_id#4736,name#4737,order_date#4738,product_name#4739,quantity#4740,unit_price#4741,line_total#4742,category#4743] parquet

== Physical Pla

---
## 14. Structured Streaming Introduction

Structured Streaming treats a live data stream as a table that is continuously appended to. It uses the same DataFrame API.

**Note**: This section demonstrates the API but uses file-based sources (suitable for local testing).

In [58]:
# Setup: Write some JSON files to simulate a stream source
import json

stream_input_dir = os.path.join(output_dir, "stream_input")
stream_output_dir = os.path.join(output_dir, "stream_output")
stream_checkpoint_dir = os.path.join(output_dir, "stream_checkpoint")

os.makedirs(stream_input_dir, exist_ok=True)

# Write sample JSON files (simulating incoming events)
events_batch1 = [
    {
        "event_id": 1,
        "user_id": 1,
        "action": "view",
        "product_id": 101,
        "timestamp": "2024-01-01 10:00:00",
    },
    {
        "event_id": 2,
        "user_id": 2,
        "action": "click",
        "product_id": 102,
        "timestamp": "2024-01-01 10:01:00",
    },
    {
        "event_id": 3,
        "user_id": 1,
        "action": "add_to_cart",
        "product_id": 101,
        "timestamp": "2024-01-01 10:02:00",
    },
    {
        "event_id": 4,
        "user_id": 3,
        "action": "view",
        "product_id": 103,
        "timestamp": "2024-01-01 10:03:00",
    },
    {
        "event_id": 5,
        "user_id": 1,
        "action": "purchase",
        "product_id": 101,
        "timestamp": "2024-01-01 10:05:00",
    },
]

events_batch2 = [
    {
        "event_id": 6,
        "user_id": 4,
        "action": "view",
        "product_id": 104,
        "timestamp": "2024-01-01 10:06:00",
    },
    {
        "event_id": 7,
        "user_id": 2,
        "action": "purchase",
        "product_id": 102,
        "timestamp": "2024-01-01 10:07:00",
    },
    {
        "event_id": 8,
        "user_id": 5,
        "action": "view",
        "product_id": 105,
        "timestamp": "2024-01-01 10:08:00",
    },
    {
        "event_id": 9,
        "user_id": 3,
        "action": "add_to_cart",
        "product_id": 103,
        "timestamp": "2024-01-01 10:09:00",
    },
    {
        "event_id": 10,
        "user_id": 5,
        "action": "click",
        "product_id": 105,
        "timestamp": "2024-01-01 10:10:00",
    },
]

# Write as JSON files
with open(os.path.join(stream_input_dir, "batch1.json"), "w") as f:
    f.writelines(json.dumps(event) + "\n" for event in events_batch1)

with open(os.path.join(stream_input_dir, "batch2.json"), "w") as f:
    f.writelines(json.dumps(event) + "\n" for event in events_batch2)

print(f"Stream input files written to: {stream_input_dir}")

Stream input files written to: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/pyspark_tutorial_output/stream_input


In [ ]:
# Define schema for streaming data
event_schema = StructType([
    StructField("event_id", IntegerType(), False),
    StructField("user_id", IntegerType(), False),
    StructField("action", StringType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
])

# Read as a stream (from JSON files)
stream_df = spark.readStream.schema(event_schema).json(stream_input_dir)

print(f"Is streaming: {stream_df.isStreaming}")
stream_df.printSchema()

Is streaming: True
root
 |-- event_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- action: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [63]:
# Apply transformations on the stream (same API as batch!)
action_counts = stream_df.groupBy("action").agg(
    F.count("*").alias("event_count"),
    F.approx_count_distinct("user_id").alias("unique_users"),
)

# Write stream output (using "complete" mode for aggregations)
query = (
    action_counts.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("action_counts")
    .option("checkpointLocation", stream_checkpoint_dir)
    .start()
)

# Wait for data to be processed
query.processAllAvailable()

# Query the in-memory table
print("=== Streaming Results: Action Counts ===")
spark.sql("SELECT * FROM action_counts ORDER BY event_count DESC").show()

# Stop the streaming query
query.stop()
print("Stream stopped.")

26/08/05 18:57:36 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/08/05 18:57:36 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


=== Streaming Results: Action Counts ===
+-----------+-----------+------------+
|     action|event_count|unique_users|
+-----------+-----------+------------+
|       view|          4|           4|
|   purchase|          2|           2|
|add_to_cart|          2|           2|
|      click|          2|           2|
+-----------+-----------+------------+

Stream stopped.


26/08/05 18:57:36 WARN DAGScheduler: Failed to cancel job group cc8aecb2-553a-474e-ae34-8729e99051ee. Cannot find active jobs for it.
26/08/05 18:57:36 WARN DAGScheduler: Failed to cancel job group cc8aecb2-553a-474e-ae34-8729e99051ee. Cannot find active jobs for it.


In [64]:
# Windowed aggregation on event time
windowed_stream = (
    stream_df
    .withWatermark("timestamp", "5 minutes")
    .groupBy(F.window("timestamp", "5 minutes", "2 minutes"), "action")
    .count()
)

# Write to memory sink
windowed_checkpoint_dir = os.path.join(output_dir, "stream_checkpoint_windowed")
query2 = (
    windowed_stream.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("windowed_actions")
    .option("checkpointLocation", windowed_checkpoint_dir)
    .start()
)

query2.processAllAvailable()

print("=== Windowed Stream Results ===")
spark.sql("""
    SELECT 
        window.start AS window_start,
        window.end AS window_end,
        action,
        count
    FROM windowed_actions
    ORDER BY window_start, action
""").show(truncate=False)

query2.stop()
print("Windowed stream stopped.")

26/08/05 18:57:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/08/05 18:57:43 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


=== Windowed Stream Results ===
+-------------------+-------------------+-----------+-----+
|window_start       |window_end         |action     |count|
+-------------------+-------------------+-----------+-----+
|2024-01-01 09:56:00|2024-01-01 10:01:00|view       |1    |
|2024-01-01 09:58:00|2024-01-01 10:03:00|add_to_cart|1    |
|2024-01-01 09:58:00|2024-01-01 10:03:00|click      |1    |
|2024-01-01 09:58:00|2024-01-01 10:03:00|view       |1    |
|2024-01-01 10:00:00|2024-01-01 10:05:00|add_to_cart|1    |
|2024-01-01 10:00:00|2024-01-01 10:05:00|click      |1    |
|2024-01-01 10:00:00|2024-01-01 10:05:00|view       |2    |
|2024-01-01 10:02:00|2024-01-01 10:07:00|add_to_cart|1    |
|2024-01-01 10:02:00|2024-01-01 10:07:00|purchase   |1    |
|2024-01-01 10:02:00|2024-01-01 10:07:00|view       |2    |
|2024-01-01 10:04:00|2024-01-01 10:09:00|purchase   |2    |
|2024-01-01 10:04:00|2024-01-01 10:09:00|view       |2    |
|2024-01-01 10:06:00|2024-01-01 10:11:00|add_to_cart|1    |
|2024-01

26/08/05 18:57:44 WARN DAGScheduler: Failed to cancel job group bb87d4ce-190a-4d03-a07f-6309081f9a60. Cannot find active jobs for it.
26/08/05 18:57:44 WARN DAGScheduler: Failed to cancel job group bb87d4ce-190a-4d03-a07f-6309081f9a60. Cannot find active jobs for it.


### Streaming Output Modes

| Mode | Description | Use Case |
|------|-------------|----------|
| **append** | Only new rows added (no updates) | Simple filters, maps |
| **complete** | Full result table every trigger | Aggregations |
| **update** | Only changed rows | Aggregations with less output |

### Streaming Sinks

| Sink | Description |
|------|-------------|
| File (Parquet/JSON/CSV) | Write to files |
| Kafka | Write to Kafka topics |
| Memory | In-memory table (testing only) |
| Console | Print to stdout (debugging) |
| foreach/foreachBatch | Custom logic per row/batch |

---
## 15. Cleanup

Always stop SparkSession when done to release resources.

In [65]:
# Clean up temporary files
import shutil

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print(f"Cleaned up: {output_dir}")

# Stop SparkSession
spark.stop()
print("SparkSession stopped. Tutorial complete!")

Cleaned up: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/pyspark_tutorial_output
SparkSession stopped. Tutorial complete!


26/08/05 18:58:36 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:1644)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:1596)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:1278)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:539)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:635)
	at java.base/java.lang.Thre

---
## Summary

### Key Takeaways

1. **SparkSession** is your entry point — always start here
2. **DataFrames** > RDDs for 95% of use cases
3. **Lazy evaluation** — transformations build a plan, actions execute it
4. **Use built-in functions** over UDFs whenever possible (Catalyst optimizer)
5. **Parquet** is the preferred file format (columnar, compressed, schema-aware)
6. **Partition wisely** — aim for 128MB per partition, 2-4 partitions per core
7. **Cache** DataFrames that are reused multiple times
8. **Broadcast joins** for small dimension tables
9. **Spark SQL** and DataFrame API have identical performance
10. **Check explain plans** to understand and optimize queries

### Next Steps

- Explore **MLlib** for machine learning pipelines
- Try **Delta Lake** for ACID transactions on data lakes
- Learn **Spark on Kubernetes** for production deployments
- Practice with larger datasets (NYC Taxi, TPC-DS benchmarks)